# Table 3: Original vs. FFCA-Pruned MLP — PNG Generator

Reads `results/ffca_vs_original_table3.csv` and saves a publication-quality PNG to `results/table3_ffca_comparison.png`.

Run the single code cell below from the repo root (or adjust `REPO_ROOT` to point to the repo).

In [2]:
import os, pathlib
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── paths ──────────────────────────────────────────────────────────────────
# REPO_ROOT = pathlib.Path(os.getcwd())          # assumes notebook runs from repo root
# If running from notebooks/, uncomment the line below instead:
REPO_ROOT = pathlib.Path(os.getcwd()).parent

CSV_PATH = REPO_ROOT / "results" / "ffca_vs_original_table3.csv"
OUT_PATH = REPO_ROOT / "results" / "table3_ffca_comparison.png"

# ── load data ──────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)

MODEL_ORDER = [
    "Measurements Only",
    "Predicted Ocean Water Levels",
    "Predicted Rainfall",
    "Predicted Gate Opening",
    "Predictions All Inputs",
]
LEADS = [3, 6, 12, 24]

orig = df[df["Variant"] == "Original"].set_index(["Model", "Lead (h)"])
ffca = df[df["Variant"] == "FFCA"].set_index(["Model", "Lead (h)"])

# ── build row data ─────────────────────────────────────────────────────────
# Each row: [model, lead, n_feat_orig, n_feat_ffca,
#            cf15_o, cf15_f, cf5_o, cf5_f, cf1_o, cf1_f,
#            rmse_o, rmse_f, r2_o, r2_f]
rows = []
model_start_row = {}   # model_name -> first row index
for model in MODEL_ORDER:
    model_start_row[model] = len(rows)
    for lead in LEADS:
        o = orig.loc[(model, lead)]
        f = ffca.loc[(model, lead)]
        rows.append([
            model, lead,
            "-",                   str(int(f["N_feat"])),
            f"{o['CF_15CM']:.1f}", f"{f['CF_15CM']:.1f}",
            f"{o['CF_5CM']:.1f}",  f"{f['CF_5CM']:.1f}",
            f"{o['CF_1CM']:.1f}",  f"{f['CF_1CM']:.1f}",
            f"{o['RMSE_cm']:.1f}", f"{f['RMSE_cm']:.1f}",
            f"{o['R2']:.2f}",      f"{f['R2']:.2f}",
        ])

# ── layout constants ───────────────────────────────────────────────────────
COL_LABELS = [
    "Model", "Lead\n(h)",
    "Feat\nOrig", "Feat\nFFCA",
    "CF 15cm\nOrig", "CF 15cm\nFFCA",
    "CF 5cm\nOrig",  "CF 5cm\nFFCA",
    "CF 1cm\nOrig",  "CF 1cm\nFFCA",
    "RMSE\nOrig",    "RMSE\nFFCA",
    "R²\nOrig",      "R²\nFFCA",
]
# relative column widths (must match len(COL_LABELS))
COL_W = [3.2, 0.65, 0.65, 0.65, 0.85, 0.85, 0.85, 0.85, 0.85, 0.85, 0.85, 0.85, 0.80, 0.80]
N_COLS = len(COL_LABELS)
N_ROWS = len(rows)

HEADER_BG = "#2c3e50"
HEADER_FG = "white"
BETTER_BG = "#d4edda"   # green  – FFCA improves
WORSE_BG  = "#f8d7da"   # red    – FFCA degrades
NEUTRAL_A = "white"
NEUTRAL_B = "#f5f5f5"   # alternating group shade
GROUP_LINE = "#2c3e50"
TEXT_COL   = "#1a252f"

# convert relative widths to x-coordinates in "data space" (0..N_COLS)
total_w = sum(COL_W)
col_x = [sum(COL_W[:i]) / total_w * N_COLS for i in range(N_COLS + 1)]
col_cx = [(col_x[i] + col_x[i+1]) / 2 for i in range(N_COLS)]

fig, ax = plt.subplots(figsize=(19, 0.52 * N_ROWS + 2.0))
ax.set_xlim(0, N_COLS)
ax.set_ylim(0, N_ROWS + 1)
ax.axis("off")

# ── header row ─────────────────────────────────────────────────────────────
for c, label in enumerate(COL_LABELS):
    rect = plt.Rectangle(
        (col_x[c], N_ROWS), col_x[c+1] - col_x[c], 1,
        facecolor=HEADER_BG, edgecolor="white", linewidth=0.5,
    )
    ax.add_patch(rect)
    ax.text(col_cx[c], N_ROWS + 0.5, label,
            ha="center", va="center", color=HEADER_FG,
            fontsize=8.0, fontweight="bold", linespacing=1.25)

# ── data rows ──────────────────────────────────────────────────────────────
# FFCA columns that can be green/red (paired with orig col immediately before)
FFCA_COLS = {5: 4, 7: 6, 9: 8, 11: 10, 13: 12}  # ffca_col: orig_col
RMSE_FFCA_COL = 11   # lower is better for RMSE

for r, row in enumerate(rows):
    y = N_ROWS - 1 - r
    model_name = row[0]
    group_idx = MODEL_ORDER.index(model_name)
    base_bg = NEUTRAL_A if group_idx % 2 == 0 else NEUTRAL_B

    for c in range(N_COLS):
        bg = base_bg

        if c in FFCA_COLS:
            orig_col = FFCA_COLS[c]
            try:
                oval = float(row[orig_col])
                fval = float(row[c])
                delta = fval - oval
                if c == RMSE_FFCA_COL:      # lower RMSE = better
                    delta = -delta
                if delta > 0.05:
                    bg = BETTER_BG
                elif delta < -0.05:
                    bg = WORSE_BG
            except (ValueError, TypeError):
                pass

        ax.add_patch(plt.Rectangle(
            (col_x[c], y), col_x[c+1] - col_x[c], 1,
            facecolor=bg, edgecolor="#cccccc", linewidth=0.3,
        ))

        # suppress model name in col 0 — rendered once per group below
        txt = "" if c == 0 else row[c]
        ax.text(col_cx[c], y + 0.5, txt,
                ha="center", va="center", fontsize=8.5, color=TEXT_COL)

# ── group dividers + model name labels ─────────────────────────────────────
for model_name, start_r in model_start_row.items():
    # horizontal divider above each group (except the very first)
    if start_r > 0:
        y_line = N_ROWS - start_r
        ax.plot([0, N_COLS], [y_line, y_line], color=GROUP_LINE, linewidth=1.4)

    y_bot = N_ROWS - (start_r + len(LEADS))
    y_top = N_ROWS - start_r
    cy = (y_bot + y_top) / 2

    ax.text(col_cx[0], cy, model_name,
            ha="center", va="center", fontsize=8.5,
            fontweight="bold", color=TEXT_COL,
            multialignment="center")

# ── outer border ───────────────────────────────────────────────────────────
ax.plot([0, N_COLS], [N_ROWS + 1, N_ROWS + 1], color=GROUP_LINE, linewidth=2.0)
ax.plot([0, N_COLS], [N_ROWS, N_ROWS],          color=GROUP_LINE, linewidth=2.0)
ax.plot([0, N_COLS], [0, 0],                    color=GROUP_LINE, linewidth=1.8)
ax.plot([0, 0],         [0, N_ROWS + 1],        color=GROUP_LINE, linewidth=1.8)
ax.plot([N_COLS, N_COLS], [0, N_ROWS + 1],      color=GROUP_LINE, linewidth=1.8)

# ── legend ─────────────────────────────────────────────────────────────────
better_patch = mpatches.Patch(facecolor=BETTER_BG, edgecolor="#888", label="FFCA better")
worse_patch  = mpatches.Patch(facecolor=WORSE_BG,  edgecolor="#888", label="FFCA worse / no change")
ax.legend(handles=[better_patch, worse_patch],
          loc="lower center", bbox_to_anchor=(0.5, -0.06),
          ncol=2, fontsize=9.5, framealpha=0.9)

# ── title ──────────────────────────────────────────────────────────────────
fig.suptitle(
    "Table 3: MLP Performance — Original Feature Set vs. FFCA-Pruned Feature Set\n"
    "(CF in %, RMSE in cm; threshold for colour coding: ±0.05 units)",
    fontsize=11.5, fontweight="bold", y=1.01, color=TEXT_COL,
)

plt.tight_layout(rect=[0, 0.05, 1, 0.99])
plt.savefig(OUT_PATH, dpi=180, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved → {OUT_PATH}")


Saved → C:\Users\jcham070\ml-miami-compound-flood-predictions\results\table3_ffca_comparison.png


C:\Users\jcham070\AppData\Local\Temp\ipykernel_18804\630732340.py:175: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
